# Per-Visit Astrometric Residual Field vs PSF Moments (v1)

**Author:** Aaron Roodman
**Date Created:** 2026-08-13
**Last Modified:** 2026-08-13
**Status:** In Progress
**Keywords:** astrometry, atmospheric turbulence, wind, PSF ellipticity, WCS residuals, optical misalignment

## Description

Study the per-visit astrometric residuals of individual LSSTCam images as a probe of the
atmospheric contribution, and compare their spatial pattern across the focal plane with the
PSF ellipticity / higher-moment pattern for the same visit.

The astrometric residual measured here is the single-frame WCS solution (built on the fixed
`astrometry_camera` distortion model + a per-visit affine) differenced against the Gaia
astrometric reference catalog, **before** the Gaussian-Process turbulence correction
(`fit_turbulence`). That residual field should carry a significant, spatially-coherent
(wind-streaked) turbulence component.

Key functionality:
1. Locate and load `preliminary_visit_image`, `single_visit_star_ref_match_astrom`, and
   `single_visit_star` for one visit.
2. Build the astrometric residual vector field (measured − Gaia, mas), **median-binned on
   CCD-aligned 2×2 focal-plane bins** (robust to non-Gaussian tails).
3. Build the PSF ellipticity field from the star second moments (`ixx/iyy/ixy`).
4. Compare the two fields for one visit, then **stack a night's in-focus visits** to separate
   the persistent (optical-misalignment) component from the visit-varying (atmospheric) one.

**Output:** per-visit residual quiver vs PSF ellipticity whisker (CCD-binned); residual
histograms; multi-visit persistent-vs-scatter maps; per-visit arrays saved to `output/`.

**Based on:** DM astrometric pipeline (`GbdesAstrometricFitTask`, `BuildCameraFromAstrometryTask`,
`fit_turbulence`), analysis_tools `refCatMatchPlots`, and the moment machinery in
`rubin-work/optatmo` (`extract_psf_moments.py`, `moments_hsm.py`). C. Saunders (DM) astrometric work.

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-08-13 | Aaron Roodman | Initial version — per-visit residual field + PSF ellipticity comparison |
| 2026-08-13 | Aaron Roodman | Median CCD-quadrant binning (2×2/CCD) + CCD grid; linear ±50 mas residual histograms; multi-visit stack (persistent vs variable) |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Analysis — Single-Visit Residual & Ellipticity](#analysis)
6. [Results & Plots — Single Visit](#results)
7. [Multi-Visit PDF & Stacked Field](#stack)
8. [Fixed Camera Model (astrometry_camera)](#astrocam)
9. [Next Steps](#next)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================
VISIT        = 2026051300022          # LSSTCam visit (day_obs 20260513, seq 22)
BUTLER_REPO  = "/repo/main"
COLLECTION   = "LSSTCam/runs/nightlyValidation/67"  # set None to auto-discover
DET_FOR_WCS  = 94                      # example detector for a WCS / visitInfo peek

# star selection (mirrors analysis_tools / optatmo)
SNR_MIN      = 50.0                    # psfFlux / psfFluxErr
MAX_RESID    = 200.0                   # mas loose clip on |residual| (blends); medians are robust
EXTEND_MAX   = 0.5                     # extendedness < this => point source

# binning / robust statistic
STAT         = "median"                # "median" or "clipmean" (3-sigma clipped mean)
HIST_RANGE   = 50.0                    # mas; linear residual-histogram half-range
NBINS_VISIT  = 2                       # per-visit map: bins per CCD axis (2 => 2x2/CCD)
VISIT_NMIN   = 8                       # min stars per bin, per-visit map
NBINS_STACK  = 16                      # stacked map: bins per CCD axis (16 => 16x16/CCD)
STACK_NMIN   = 3                       # min pooled stars per bin, stacked map

# multi-visit stack
STACK_SAME_BAND  = True                # restrict the stack to VISIT's band
DEMEAN_PER_VISIT = True                # subtract each visit's median ΔX/ΔY before pooling
MAX_STACK_VISITS = 40                  # cap (each visit reads per-detector WCS -> minutes)

OUTPUT_DIR   = "../output"             # gitignored; symlinked to scratch on RSP


<a id='setup'></a>
## Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.units as u

from lsst.daf.butler import Butler, CollectionType
from lsst.obs.lsst import LsstCam
import lsst.afw.cameraGeom as cg
from lsst.geom import Point2D, SpherePoint, degrees

# repo-root imports
sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting
setup_plotting()

camera = LsstCam.getCamera()           # full LSSTCam geometry (PIXELS <-> FOCAL_PLANE)
os.makedirs(OUTPUT_DIR, exist_ok=True)

<a id='functions'></a>
## Helper Functions

In [ ]:
def resolve_collection(butler, visit, dstypes):
    """Newest CHAINED run holding all `dstypes` for `visit` (fast; avoids "*")."""
    import re
    chains = list(butler.registry.queryCollections(collectionTypes=CollectionType.CHAINED))
    cand = sorted(c for c in chains if re.search(r"LSSTCam/(runs|nightly|DRP)", c, re.I))
    runs = {}
    for dt in dstypes:
        runs[dt] = sorted({r.run for r in butler.registry.queryDatasets(
            dt, collections=cand, findFirst=False,
            where=f"instrument='LSSTCam' AND visit={visit}")})
    common = [r for r in runs[dstypes[0]] if all(r in runs[d] for d in dstypes)]
    return (common[-1] if common else None), runs


def make_wcs_getter(butler, visit):
    """Cached per-detector WCS via cheap component read (no pixels loaded)."""
    cache = {}
    def get_wcs(det):
        det = int(det)
        if det not in cache:
            cache[det] = butler.get("preliminary_visit_image.wcs",
                                    instrument="LSSTCam", visit=visit, detector=det)
        return cache[det]
    return get_wcs


def pix_to_fp(det, x, y):
    """Detector pixel (x, y) -> nominal focal-plane position (mm)."""
    p = camera[int(det)].getTransform(cg.PIXELS, cg.FOCAL_PLANE).applyForward(Point2D(x, y))
    return p.getX(), p.getY()


def ellipticity(ixx, iyy, ixy):
    """Second-moment ellipticity: e1=(ixx-iyy)/T, e2=2ixy/T, T=ixx+iyy."""
    T = ixx + iyy
    return (ixx - iyy) / T, 2 * ixy / T, T


def nmad(v):
    """Normalized median absolute deviation (robust sigma)."""
    v = np.asarray(v, float)
    return 1.4826 * np.median(np.abs(v - np.median(v)))


def robust_stats(v):
    """(n, median, mean, std, nmad-sigma) for a 1-D array."""
    v = np.asarray(v, float)
    return len(v), np.median(v), np.mean(v), np.std(v), nmad(v)


def clipped_mean(v, nsig=3, iters=3):
    """Iterative n-sigma (NMAD) clipped mean."""
    v = np.asarray(v, float)
    for _ in range(iters):
        m, s = np.median(v), nmad(v)
        keep = np.abs(v - m) <= nsig * max(s, 1e-9)
        if keep.all() or keep.sum() < 3:
            break
        v = v[keep]
    return v.mean()


def ccd_bin(det, ix, iy, u_, v_, nper=2, reduce=np.median, nmin=8):
    """Bin a vector field into an nper x nper grid per CCD, keyed to CCD geometry.

    Each star is assigned to one of nper*nper cells of its detector (uniform split
    in pixel space); the bin position is the nominal focal-plane center of that
    cell, and the value is `reduce` applied to the stars in it. nper=2 => 2x2/CCD;
    nper=16 => 16x16/CCD.

    Returns (keys, X, Y, U, V, N) where keys[i] = (det, bx, by).
    """
    det = np.asarray(det).astype(int)
    ix = np.asarray(ix, float); iy = np.asarray(iy, float)
    u_ = np.asarray(u_, float); v_ = np.asarray(v_, float)
    bb = {d: camera[int(d)].getBBox() for d in np.unique(det)}
    minx = np.array([bb[d].getMinX() for d in det]); w = np.array([bb[d].getWidth()  for d in det])
    miny = np.array([bb[d].getMinY() for d in det]); h = np.array([bb[d].getHeight() for d in det])
    bx = np.clip(((ix - minx) / w * nper).astype(int), 0, nper - 1)
    by = np.clip(((iy - miny) / h * nper).astype(int), 0, nper - 1)
    df = pd.DataFrame({"det": det, "bx": bx, "by": by, "u": u_, "v": v_})
    keys, X, Y, U, V, N = [], [], [], [], [], []
    for (d, ax_, ay_), g in df.groupby(["det", "bx", "by"]):
        if len(g) < nmin:
            continue
        b = bb[d]
        px = b.getMinX() + b.getWidth()  * (ax_ + 0.5) / nper
        py = b.getMinY() + b.getHeight() * (ay_ + 0.5) / nper
        fx, fy = pix_to_fp(int(d), px, py)
        keys.append((int(d), int(ax_), int(ay_)))
        X.append(fx); Y.append(fy)
        U.append(reduce(g.u.values)); V.append(reduce(g.v.values)); N.append(len(g))
    return keys, np.array(X), np.array(Y), np.array(U), np.array(V), np.array(N)


def draw_ccd_grid(ax, lw=0.3, color="0.75"):
    """Overlay science-CCD outlines and their 2x2 quadrant midlines (focal-plane mm)."""
    for det in camera:
        if det.getType() != cg.DetectorType.SCIENCE:
            continue
        bb = det.getBBox()
        tr = det.getTransform(cg.PIXELS, cg.FOCAL_PLANE)
        f = lambda x, y: (lambda p: (p.getX(), p.getY()))(tr.applyForward(Point2D(x, y)))
        x0, y0, x1, y1 = bb.getMinX(), bb.getMinY(), bb.getMaxX(), bb.getMaxY()
        xc, yc = bb.getCenterX(), bb.getCenterY()
        outline = [f(x0, y0), f(x1, y0), f(x1, y1), f(x0, y1), f(x0, y0)]
        ax.plot([p[0] for p in outline], [p[1] for p in outline], color=color, lw=lw, zorder=0)
        for a, b in [((xc, y0), (xc, y1)), ((x0, yc), (x1, yc))]:
            (px0, py0), (px1, py1) = f(*a), f(*b)
            ax.plot([px0, px1], [py0, py1], color=color, lw=lw*0.7, zorder=0)


def extract_visit_stars(butler, visit, get_wcs=None):
    """Per-visit matched stars: measured-Gaia residuals (mas) + pixel/detector/FP position.

    Returns a dict of numpy arrays for the clipped field (det, ix, iy, fx, fy, dX, dY)
    plus the pre-clip quality residuals (dX_q, dY_q) for histograms.
    """
    if get_wcs is None:
        get_wcs = make_wcs_getter(butler, visit)
    rm = butler.get("single_visit_star_ref_match_astrom",
                    instrument="LSSTCam", visit=visit).to_pandas()
    cosd = np.cos(np.deg2rad(rm.coord_dec_target.to_numpy()))
    dX = ((rm.coord_ra_target  - rm.ra_ref ).to_numpy() * cosd * u.deg).to(u.mas).value
    dY = ((rm.coord_dec_target - rm.dec_ref).to_numpy()         * u.deg).to(u.mas).value
    qual = ((rm.extendedness_target < EXTEND_MAX)
            & (rm.psfFlux_target / rm.psfFluxErr_target > SNR_MIN)
            & ~rm.psfFlux_flag_target & ~rm.centroid_flag_target).to_numpy()
    qual &= np.isfinite(dX) & np.isfinite(dY)
    det = rm.detector_target.to_numpy()[qual]
    ra  = rm.coord_ra_target.to_numpy()[qual]
    dec = rm.coord_dec_target.to_numpy()[qual]
    dXq, dYq = dX[qual], dY[qual]
    ix = np.empty(len(det)); iy = np.empty(len(det))
    fx = np.empty(len(det)); fy = np.empty(len(det))
    for i, (d, r, c) in enumerate(zip(det, ra, dec)):
        pix = get_wcs(int(d)).skyToPixel(SpherePoint(r, c, degrees))
        ix[i], iy[i] = pix.getX(), pix.getY()
        fx[i], fy[i] = pix_to_fp(int(d), ix[i], iy[i])
    clip = np.hypot(dXq, dYq) < MAX_RESID
    return dict(det=det[clip], ix=ix[clip], iy=iy[clip], fx=fx[clip], fy=fy[clip],
                dX=dXq[clip], dY=dYq[clip], dX_q=dXq, dY_q=dYq, n_qual=int(qual.sum()))

<a id='data'></a>
## Data Access

In [ ]:
butler = Butler(BUTLER_REPO)

# visit metadata (band via expandDataId — not a stored field on the visit record)
did  = butler.registry.expandDataId(instrument="LSSTCam", visit=VISIT)
vrec = did.records["visit"]
BAND, PHYS, DAYOBS = did["band"], did["physical_filter"], vrec.day_obs
print(f"visit {VISIT}: band={BAND} filter={PHYS} day_obs={DAYOBS}")

# collection: use the parameter, or auto-discover
DSTYPES = ["single_visit_star_ref_match_astrom", "single_visit_star", "preliminary_visit_image"]
if COLLECTION is None:
    COLLECTION, runs = resolve_collection(butler, VISIT, DSTYPES)
    print("discovered runs:", {k: v[-2:] for k, v in runs.items()})
assert COLLECTION, "no collection holds these products for this visit"
print("USING COLLECTION:", COLLECTION)

butler  = Butler(BUTLER_REPO, collections=[COLLECTION])
get_wcs = make_wcs_getter(butler, VISIT)
REDUCE  = np.median if STAT == "median" else clipped_mean

# example WCS + visitInfo (boresight / rotator / parallactic angle for the wind frame)
pvi = butler.get("preliminary_visit_image",
                 instrument="LSSTCam", visit=VISIT, detector=DET_FOR_WCS)
vi = pvi.getInfo().getVisitInfo()
print("boresight :", vi.getBoresightRaDec())
print("rotAngle  :", vi.getBoresightRotAngle().asDegrees(), "deg")
print("parAngle  :", vi.getBoresightParAngle().asDegrees(), "deg")

<a id='analysis'></a>
## Analysis — Single-Visit Residual & Ellipticity

In [ ]:
# ---- per-visit astrometric residuals (measured - Gaia), pre-GP ----
S = extract_visit_stars(butler, VISIT, get_wcs)
det_f, ix_f, iy_f = S["det"], S["ix"], S["iy"]
fx, fy, dRA, dDec = S["fx"], S["fy"], S["dX"], S["dY"]
dX_q, dY_q        = S["dX_q"], S["dY_q"]          # pre-clip quality stars (histograms)

print(f"{S['n_qual']} quality stars; {len(dRA)} kept after |resid|<{MAX_RESID} mas")
print(f"median   ΔX={np.median(dRA):+.2f}  ΔY={np.median(dDec):+.2f} mas   "
      f"(mean {np.mean(dRA):+.2f}/{np.mean(dDec):+.2f})")
print(f"NMAD     ΔX={nmad(dRA):.2f}  ΔY={nmad(dDec):.2f} mas")

In [ ]:
# ---- residual distribution diagnostics: LINEAR, +/- HIST_RANGE mas ----
# mean (dotted) vs median (dashed) makes the tail-driven bias obvious -> use medians.
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
for a, d, lab, col in [(ax[0], dX_q, "ΔX  (RA·cosδ)", "C0"),
                       (ax[1], dY_q, "ΔY  (Dec)",     "C3")]:
    n, med, mean, std, s = robust_stats(d)
    a.hist(d, bins=100, range=(-HIST_RANGE, HIST_RANGE), color=col, alpha=0.8)
    a.axvline(med,  ls="--", c="k",   lw=1.5, label=f"median={med:+.2f}")
    a.axvline(mean, ls=":",  c="0.3", lw=1.5, label=f"mean={mean:+.2f}")
    a.axvline(0, c="0.6", lw=0.8)
    a.set_title(f"{lab}   NMAD={s:.2f} mas  (n={n})")
    a.set_xlabel("residual (mas)"); a.legend(fontsize=9)

ax[2].hexbin(dX_q, dY_q, gridsize=60, cmap="viridis",
             extent=(-HIST_RANGE, HIST_RANGE, -HIST_RANGE, HIST_RANGE))
ax[2].axhline(0, c="w", lw=0.5); ax[2].axvline(0, c="w", lw=0.5)
ax[2].set_aspect("equal"); ax[2].set_xlabel("ΔX (mas)"); ax[2].set_ylabel("ΔY (mas)")
ax[2].set_title("ΔX vs ΔY (density)")
plt.savefig(f"{OUTPUT_DIR}/astrometry_residual_hist_{VISIT}.png", dpi=120)
plt.show()

r = np.hypot(dX_q, dY_q)
print(f"|resid| percentiles (mas): 50%={np.percentile(r,50):.1f}  "
      f"90%={np.percentile(r,90):.1f}  99%={np.percentile(r,99):.1f}  max={r.max():.1f}")

In [ ]:
# ---- PSF ellipticity field for the SAME visit (star 2nd moments) ----
# single_visit_star stores 2nd moments as ixx/iyy/ixy (same columns optatmo reads).
svs = butler.get("single_visit_star", instrument="LSSTCam", visit=VISIT).to_pandas()
print("shape/moment cols:", [c for c in svs.columns
      if any(k in c.lower() for k in ("ixx","iyy","ixy","moment"))][:20])

snr_s = svs.psfFlux / svs.psfFluxErr
ok = (svs.get("detect_isPrimary", True) & (svs.extendedness < EXTEND_MAX)
      & (snr_s > SNR_MIN) & ~svs.pixelFlags_saturated & ~svs.pixelFlags_bad
      & np.isfinite(svs.ixx) & np.isfinite(svs.ixy) & (svs.ixx + svs.iyy > 0)).to_numpy()

e1, e2, T = ellipticity(svs.ixx.to_numpy(), svs.iyy.to_numpy(), svs.ixy.to_numpy())
e1, e2 = e1[ok], e2[ok]
sdet, six, siy = (svs.detector.to_numpy()[ok], svs.x.to_numpy()[ok], svs.y.to_numpy()[ok])
sfp = np.array([pix_to_fp(d, x, y) for d, x, y in zip(sdet, six, siy)])
print(f"{ok.sum()} PSF stars; median |e|={np.median(np.hypot(e1,e2)):.3f}")

<a id='results'></a>
## Results & Plots — Single Visit

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 7.5), sharex=True, sharey=True)

# left: astrometric residual, MEDIAN per CCD cell (NBINS_VISIT per axis)
_, X, Y, U, V, N = ccd_bin(det_f, ix_f, iy_f, dRA, dDec,
                           nper=NBINS_VISIT, reduce=REDUCE, nmin=VISIT_NMIN)
draw_ccd_grid(ax[0])
q = ax[0].quiver(X, Y, U, V, np.hypot(U, V), cmap="viridis", angles="xy", pivot="mid")
plt.colorbar(q, ax=ax[0], fraction=0.046, label=f"|{STAT} residual| (mas)")
ax[0].set_title(f"astrometric residual (measured − Gaia)\nvisit {VISIT}  {BAND}-band  "
                f"({STAT}, {NBINS_VISIT}×{NBINS_VISIT}/CCD, pre-GP)")

# right: PSF ellipticity whisker, MEDIAN per CCD cell
_, Xe, Ye, E1, E2, Ne = ccd_bin(sdet, six, siy, e1, e2,
                                nper=NBINS_VISIT, reduce=REDUCE, nmin=VISIT_NMIN)
emag, ang = np.hypot(E1, E2), 0.5*np.arctan2(E2, E1)
draw_ccd_grid(ax[1])
w = ax[1].quiver(Xe, Ye, emag*np.cos(ang), emag*np.sin(ang), emag, cmap="magma",
                 headwidth=1, headlength=0, pivot="mid", angles="xy")
plt.colorbar(w, ax=ax[1], fraction=0.046, label="|e|")
ax[1].set_title(f"PSF ellipticity whisker\nvisit {VISIT}  {BAND}-band  ({STAT}, 2×2/CCD)")

for a in ax:
    a.set_aspect("equal"); a.set_xlabel("focal-plane x (mm)")
ax[0].set_ylabel("focal-plane y (mm)")
plt.savefig(f"{OUTPUT_DIR}/astrometry_residual_vs_ellipticity_{VISIT}.png", dpi=120)
plt.show()

In [ ]:
# persist the per-visit residual field for downstream (structure fn / 2-pt / wind)
np.savez(f"{OUTPUT_DIR}/astrometry_residual_field_{VISIT}.npz",
         det=det_f, ix=ix_f, iy=iy_f, fx=fx, fy=fy, dRA=dRA, dDec=dDec,
         sfp_x=sfp[:, 0], sfp_y=sfp[:, 1], e1=e1, e2=e2,
         band=str(BAND), rot_deg=vi.getBoresightRotAngle().asDegrees(),
         par_deg=vi.getBoresightParAngle().asDegrees())
print("saved residual field npz")

<a id='stack'></a>
## Multi-Visit PDF & Stacked Field

Loop the night's in-focus (processed) visits and write a **multi-page PDF**:

- **one page per visit** — the astrometric residual arrows at `NBINS_VISIT`×`NBINS_VISIT`
  per CCD (coarse: a single visit has only a handful of stars per CCD);
- **a final stacked page** — all visits' stars **pooled** and binned at
  `NBINS_STACK`×`NBINS_STACK` per CCD (16×16). Pooling gives ~N_visits× more stars, so the
  finer bins fill in; each visit is demeaned first (`DEMEAN_PER_VISIT`) so per-visit pointing
  offsets don't blur the pattern. The stacked, focal-plane-static pattern is the
  **optical-misalignment / residual-camera** candidate.

Each visit reads per-detector WCS, so this loops over up to `MAX_STACK_VISITS` visits.

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

# night's processed science visits (those with a ref-match table on this day)
where = f"instrument='LSSTCam' AND visit.day_obs={DAYOBS}"
if STACK_SAME_BAND:
    where += f" AND band='{BAND}'"
night_visits = sorted({r.dataId["visit"] for r in butler.registry.queryDatasets(
    "single_visit_star_ref_match_astrom", collections=[COLLECTION],
    findFirst=True, where=where)})[:MAX_STACK_VISITS]
print(f"{len(night_visits)} visits:", night_visits)

def arrow_page(pdf, X, Y, U, V, title, figsize=(9, 8.5)):
    fig, a = plt.subplots(figsize=figsize)
    draw_ccd_grid(a)
    q = a.quiver(X, Y, U, V, np.hypot(U, V), cmap="viridis", angles="xy", pivot="mid")
    fig.colorbar(q, ax=a, fraction=0.046, label=f"|{STAT} residual| (mas)")
    a.set_aspect("equal"); a.set_title(title)
    a.set_xlabel("focal-plane x (mm)"); a.set_ylabel("focal-plane y (mm)")
    pdf.savefig(fig)
    return fig

pdf_path = f"{OUTPUT_DIR}/astrometry_residual_arrows_{DAYOBS}_{BAND}.pdf"
pool = {k: [] for k in ("det", "ix", "iy", "dX", "dY")}
with PdfPages(pdf_path) as pdf:
    for j, vv in enumerate(night_visits):
        try:
            Sv = extract_visit_stars(butler, vv)       # own WCS cache per visit
        except Exception as e:
            print(f"  skip {vv}: {e}"); continue
        dx, dy = Sv["dX"], Sv["dY"]
        if DEMEAN_PER_VISIT:                            # remove per-visit pointing offset
            dx = dx - np.median(dx); dy = dy - np.median(dy)
        _, X, Y, U, V, N = ccd_bin(Sv["det"], Sv["ix"], Sv["iy"], dx, dy,
                                   nper=NBINS_VISIT, reduce=REDUCE, nmin=VISIT_NMIN)
        fig = arrow_page(pdf, X, Y, U, V,
                         f"visit {vv}  {BAND}-band  ({NBINS_VISIT}×{NBINS_VISIT}/CCD"
                         + (", demeaned)" if DEMEAN_PER_VISIT else ")"))
        plt.close(fig)
        for k, arr in [("det", Sv["det"]), ("ix", Sv["ix"]), ("iy", Sv["iy"]),
                       ("dX", dx), ("dY", dy)]:
            pool[k].append(arr)
        print(f"  [{j+1}/{len(night_visits)}] visit {vv}: {len(dx)} stars, {len(X)} bins")

    # ---- stacked page: pool all stars, fine bins ----
    P = {k: np.concatenate(v) for k, v in pool.items()}
    _, SX, SY, SU, SV, SN = ccd_bin(P["det"], P["ix"], P["iy"], P["dX"], P["dY"],
                                    nper=NBINS_STACK, reduce=REDUCE, nmin=STACK_NMIN)
    fig = arrow_page(pdf, SX, SY, SU, SV,
                     f"STACKED {len(night_visits)} visits  {BAND}-band  "
                     f"({NBINS_STACK}×{NBINS_STACK}/CCD, {STAT}, demeaned)",
                     figsize=(11, 10))
    plt.show()   # also show the stacked page inline

print(f"wrote {pdf_path}  ({len(night_visits)} per-visit pages + 1 stacked; "
      f"{len(SX)} stacked bins)")
np.savez(f"{OUTPUT_DIR}/astrometry_stack_{DAYOBS}_{BAND}.npz",
         px=SX, py=SY, pu=SU, pv=SV, n=SN,
         visits=np.array(night_visits), band=str(BAND), nbins_ccd=NBINS_STACK)
print("saved stacked field npz")

<a id='astrocam'></a>
## Fixed Camera Model (astrometry_camera)

`astrometry_camera` is the static per-detector distortion model produced by
`BuildCameraFromAstrometryTask` (subtask of `GbdesAstrometricFitTask`) from the gbdes joint
fit. `calibrateImage` runs with **`useButlerCamera: True`** in nightly-validation, so the
per-visit WCS is initialized from boresight + this fixed camera model and only a per-visit
**affine** is fit on top — i.e. the static camera distortion is already removed from the
residuals above. It is a calibration dataset, one per band.

Two caveats surfaced by the checks below: (1) the model covers a subset of the 189 science
CCDs — detectors **not** in it fall back to nominal geometry in `calibrateImage`, so their
residuals still contain uncorrected distortion (candidates to flag/exclude); (2) a single
detector-center offset vs nominal is dominated by a global term the per-visit **affine
absorbs**, so it is not meaningful — the spatially-varying (mean-subtracted) distortion is.

In [ ]:
# locate astrometry_camera FAST: restrict the collection pattern (never "*").
cam_ref = list(butler.registry.queryDatasets(
    "astrometry_camera", collections="*astrometry_camera*", findFirst=False,
    where=f"instrument='LSSTCam' AND physical_filter='{PHYS}'"))[0]
astro_cam = butler.get(cam_ref)
print("astrometry_camera run:", cam_ref.run)

# ---- detector coverage: which science CCDs fell back to nominal geometry? ----
sci      = {d.getId() for d in camera if d.getType() == cg.DetectorType.SCIENCE}
present  = {d.getId() for d in astro_cam}
missing  = sorted(sci - present)
print(f"{len(present & sci)}/{len(sci)} science CCDs in astrometry_camera; "
      f"{len(missing)} fall back to nominal:", [camera[m].getName() for m in missing])

# ---- was useButlerCamera on? (discover the config dataset type/collection) ----
cfg_types = sorted(t.name for t in butler.registry.queryDatasetTypes()
                   if "calibrateImage" in t.name and t.name.endswith("_config"))
print("calibrateImage config types:", cfg_types)
cfg = None
for ct in cfg_types:
    refs = list(butler.registry.queryDatasets(ct, collections=[COLLECTION], findFirst=True))
    if refs:
        cfg = butler.get(refs[0])
        print(f"{ct}.useButlerCamera =", cfg.useButlerCamera,
              "| wcsFitter =", type(cfg.astrometry.wcsFitter.target).__name__)
        break
if cfg is None:
    print("no calibrateImage config in this chain — pipeline sets useButlerCamera: True")

# ---- meaningful distortion diff: field-angle offset at each CCD center, ----
# ---- mean removed (the constant is affine-absorbed per visit) ----
dxy = np.array([
    [(astro_cam[d].getTransform(cg.PIXELS, cg.FIELD_ANGLE)
        .applyForward(Point2D(camera[d].getBBox().getCenterX(),
                              camera[d].getBBox().getCenterY())).getX()
      - camera[d].getTransform(cg.PIXELS, cg.FIELD_ANGLE)
        .applyForward(Point2D(camera[d].getBBox().getCenterX(),
                              camera[d].getBBox().getCenterY())).getX()),
     (astro_cam[d].getTransform(cg.PIXELS, cg.FIELD_ANGLE)
        .applyForward(Point2D(camera[d].getBBox().getCenterX(),
                              camera[d].getBBox().getCenterY())).getY()
      - camera[d].getTransform(cg.PIXELS, cg.FIELD_ANGLE)
        .applyForward(Point2D(camera[d].getBBox().getCenterX(),
                              camera[d].getBBox().getCenterY())).getY())]
    for d in sorted(present & sci)]) * (180/np.pi) * 3600      # arcsec
mean_off = dxy.mean(axis=0)
resid    = dxy - mean_off
print(f"global mean offset (affine-absorbed): "
      f"{mean_off[0]:+.1f}, {mean_off[1]:+.1f} arcsec")
print(f"spatially-varying distortion diff (RMS about mean): "
      f"{np.hypot(resid[:,0].std(), resid[:,1].std()):.3f} arcsec")

<a id='next'></a>
## Next Steps

- **Rigorous higher moments (3rd/4th).** Reuse `rubin-work/optatmo/code/extract_psf_moments.py`
  (cuts stamps off `preliminary_visit_image`, PIFF-port HSM `e0,e1,e2,M21,M12,M30,M03,...`) and
  join on the CCD-quadrant grid to compare the 3rd-moment pattern with the residual field.
- **Wind test.** Rotate `(dRA, dDec)` into alt-az via the saved `rot_deg`/`par_deg`, then look
  for a preferred elongation axis and the 2-point correlation (E/B split, as `fit_turbulence`).
- **Single-visit minus persistent.** Subtract the stacked persistent field from each visit to
  isolate the atmospheric residual, and correlate it spatially with the PSF-moment field.